# 🩺 AI-Based Skin Disease Detection System
### Based on Thesis: Deep Learning Embedding with Classical Machine Learning for Accurate Skin Disease Detection
**Author:** Inder Dev | Chandigarh University

---
**Models Used:** Xception, MobileNetV3, DenseNet (Transfer Learning)

**Dataset:** HAM10000 + DermNet + SD-198

**XAI:** Grad-CAM Explainability

**Classes:** Melanoma, Nevus, BCC, Actinic Keratosis, Benign Keratosis, Dermatofibroma, Vascular, SCC, Psoriasis, Eczema

## 📦 Step 1: Install Required Libraries

In [ ]:
!pip install tensorflow keras matplotlib seaborn scikit-learn opencv-python Pillow grad-cam kaggle -q
print('✅ All libraries installed successfully!')

## 📚 Step 2: Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import Xception, MobileNetV3Large, DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy

print(f'✅ TensorFlow Version: {tf.__version__}')
print(f'✅ GPU Available: {tf.config.list_physical_devices("GPU")}')

## 📂 Step 3: Download HAM10000 Dataset from Kaggle

In [ ]:
# ✅ Upload your kaggle.json API key first
from google.colab import files
print('Please upload your kaggle.json file:')
uploaded = files.upload()

# Setup Kaggle credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download HAM10000 dataset
!kaggle datasets download -d kmader/skin-lesion-analysis-toward-melanoma-detection
!unzip -q skin-lesion-analysis-toward-melanoma-detection.zip -d /content/ham10000
print('✅ Dataset downloaded and extracted!')

## ⚙️ Step 4: Configuration & Constants

In [ ]:
# ─── CONFIGURATION ───────────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 30
LEARNING_RATE = 0.001
NUM_CLASSES  = 10
SEED         = 42

# 10 Skin Disease Classes (from thesis)
CLASS_NAMES = [
    'Melanoma',
    'Nevus',
    'Basal Cell Carcinoma',
    'Actinic Keratosis',
    'Benign Keratosis',
    'Dermatofibroma',
    'Vascular Lesions',
    'Squamous Cell Carcinoma',
    'Psoriasis',
    'Eczema'
]

# HAM10000 label mapping
LABEL_MAP = {
    'mel'  : 'Melanoma',
    'nv'   : 'Nevus',
    'bcc'  : 'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratosis',
    'bkl'  : 'Benign Keratosis',
    'df'   : 'Dermatofibroma',
    'vasc' : 'Vascular Lesions',
    'scc'  : 'Squamous Cell Carcinoma',
    'psc'  : 'Psoriasis',
    'ecz'  : 'Eczema'
}

DATA_DIR = '/content/ham10000'
print('✅ Configuration set!')
print(f'   Image Size : {IMG_SIZE}x{IMG_SIZE}')
print(f'   Batch Size : {BATCH_SIZE}')
print(f'   Epochs     : {EPOCHS}')
print(f'   Classes    : {NUM_CLASSES}')

## 🗂️ Step 5: Load & Explore Dataset

In [ ]:
# Load metadata CSV
metadata_path = '/content/ham10000/HAM10000_metadata.csv'
df = pd.read_csv(metadata_path)

print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nClass Distribution:')
print(df['dx'].value_counts())

# Map short labels to full names
df['disease_name'] = df['dx'].map(LABEL_MAP)
df['disease_name'].fillna(df['dx'], inplace=True)

df.head()

## 📊 Step 6: Dataset Visualization (Bar Chart + Pie Chart)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ── Bar Chart ──
class_counts = df['dx'].value_counts()
colors_bar = plt.cm.Blues(np.linspace(0.4, 0.9, len(class_counts)))
axes[0].bar(class_counts.index, class_counts.values, color=colors_bar, edgecolor='black', linewidth=0.5)
axes[0].set_title('Dataset Class Distribution (Bar Chart)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classes', fontsize=12)
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9)

# ── Pie Chart ──
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(class_counts)))
axes[1].pie(class_counts.values,
            labels=class_counts.index,
            autopct='%1.1f%%',
            colors=colors_pie,
            startangle=140,
            pctdistance=0.85)
axes[1].set_title('Class Distribution Pie Chart', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dataset distribution charts saved!')

## 🖼️ Step 7: Build Image Dataset with Paths

In [ ]:
# Find all image files
image_dirs = [
    '/content/ham10000/HAM10000_images_part_1',
    '/content/ham10000/HAM10000_images_part_2'
]

# Build image_id -> path mapping
image_path_dict = {}
for folder in image_dirs:
    if os.path.exists(folder):
        for fname in os.listdir(folder):
            img_id = fname.split('.')[0]
            image_path_dict[img_id] = os.path.join(folder, fname)

df['path'] = df['image_id'].map(image_path_dict)
df.dropna(subset=['path'], inplace=True)

print(f'✅ Total images found: {len(df)}')
print(f'Sample paths:')
print(df['path'].head(3).tolist())

## 🔀 Step 8: Train / Validation / Test Split (70/15/15)

In [ ]:
# Encode labels
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['dx'])

# One-hot encode
from tensorflow.keras.utils import to_categorical
y = to_categorical(df['label_encoded'], num_classes=len(le.classes_))

# Split: 70% train, 15% val, 15% test
X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    df, y, test_size=0.30, stratify=df['label_encoded'], random_state=SEED)

X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df, y_temp, test_size=0.50, stratify=X_temp_df['label_encoded'], random_state=SEED)

print(f'✅ Train samples      : {len(X_train_df)}')
print(f'✅ Validation samples : {len(X_val_df)}')
print(f'✅ Test samples       : {len(X_test_df)}')

## 🔄 Step 9: Data Preprocessing & Augmentation

In [ ]:
def load_and_preprocess_image(path, img_size=IMG_SIZE):
    """Load image, resize and normalize pixel values to [0,1]"""
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    img = img.astype(np.float32) / 255.0
    return img

def load_dataset(df_subset, img_size=IMG_SIZE):
    """Load all images for a dataframe subset"""
    images = []
    for path in df_subset['path']:
        try:
            img = load_and_preprocess_image(path, img_size)
            images.append(img)
        except Exception as e:
            images.append(np.zeros((img_size, img_size, 3), dtype=np.float32))
    return np.array(images)

print('Loading images... (this may take a few minutes)')
X_train = load_dataset(X_train_df)
X_val   = load_dataset(X_val_df)
X_test  = load_dataset(X_test_df)

print(f'✅ X_train shape: {X_train.shape}')
print(f'✅ X_val shape  : {X_val.shape}')
print(f'✅ X_test shape : {X_test.shape}')

In [ ]:
# ── Data Augmentation (Rotation, Flip, Zoom, Brightness) ──
train_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator()  # No augmentation for val/test

train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True, seed=SEED)
val_generator   = val_datagen.flow(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)

print('✅ Data Augmentation configured!')
print('   Rotation: 30°  | Flip: H+V | Zoom: 20% | Brightness: ±20%')

## 🏗️ Step 10: Build Models (Xception / MobileNetV3 / DenseNet121)

In [ ]:
def build_model(base_model_name='Xception', num_classes=NUM_CLASSES, img_size=IMG_SIZE):
    """
    Build Transfer Learning model with:
    - Pre-trained backbone (frozen initially)
    - Custom classification head
    - BatchNorm + Dropout regularization
    """
    input_shape = (img_size, img_size, 3)

    # ── Select Base Model ──
    if base_model_name == 'Xception':
        base = Xception(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name == 'MobileNetV3':
        base = MobileNetV3Large(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name == 'DenseNet121':
        base = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)
    else:
        raise ValueError(f'Unknown model: {base_model_name}')

    # Freeze base layers initially
    base.trainable = False

    # ── Classification Head ──
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=output, name=base_model_name)

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=CategoricalCrossentropy(),
        metrics=['accuracy']
    )

    return model, base

# Build Xception model (primary model)
model, base_model = build_model('Xception')
model.summary()
print(f'\n✅ Xception model built | Total params: {model.count_params():,}')

## 🚂 Step 11: Train Phase 1 (Frozen Base)

In [ ]:
# ── Callbacks ──
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model_phase1.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

print('🚂 Phase 1 Training: Frozen base layers...')
history1 = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=15,
    validation_data=val_generator,
    validation_steps=len(X_val) // BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
print('✅ Phase 1 training complete!')

## 🔓 Step 12: Fine-Tuning Phase 2 (Unfreeze Top Layers)

In [ ]:
# Unfreeze top 30 layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE / 10),
    loss=CategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks_ft = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_model_finetune.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

print('🔓 Phase 2 Fine-tuning: Top 30 layers unfrozen...')
history2 = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=len(X_val) // BATCH_SIZE,
    callbacks=callbacks_ft,
    verbose=1
)
print('✅ Fine-tuning complete!')

## 📈 Step 13: Training Curves Visualization

In [ ]:
# Combine both training histories
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

history = combine_histories(history1, history2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Accuracy Plot ──
axes[0].plot(history['accuracy'], label='Training Accuracy', color='blue', linewidth=2)
axes[0].plot(history['val_accuracy'], label='Validation Accuracy', color='orange',
             linewidth=2, linestyle='--')
axes[0].axvline(x=len(history1.history['accuracy'])-1, color='red',
                linestyle=':', label='Fine-tuning Start')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Loss Plot ──
axes[1].plot(history['loss'], label='Training Loss', color='blue', linewidth=2)
axes[1].plot(history['val_loss'], label='Validation Loss', color='orange',
             linewidth=2, linestyle='--')
axes[1].axvline(x=len(history1.history['loss'])-1, color='red',
                linestyle=':', label='Fine-tuning Start')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training & Validation Performance', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved!')

## 🧪 Step 14: Model Evaluation on Test Set

In [ ]:
print('🧪 Evaluating on Test Set...')
y_pred_proba = model.predict(X_test, verbose=1)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = np.argmax(y_test, axis=1)

# ── Metrics ──
acc       = accuracy_score(y_true, y_pred)
report    = classification_report(y_true, y_pred,
                                  target_names=le.classes_,
                                  output_dict=True)
precision = report['weighted avg']['precision']
recall    = report['weighted avg']['recall']
f1        = report['weighted avg']['f1-score']

print('\n' + '='*55)
print('         📊 OVERALL MODEL PERFORMANCE')
print('='*55)
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {precision*100:.2f}%')
print(f'  Recall    : {recall*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print('='*55)
print('\n📋 Per-Class Classification Report:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

## 🔢 Step 15: Confusion Matrix

In [ ]:
cm_matrix = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 11))
sns.heatmap(cm_matrix,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            linewidths=0.5,
            linecolor='gray')
plt.title('Confusion Matrix - Test Dataset (Multi-class)', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Class', fontsize=13)
plt.xlabel('Predicted Class', fontsize=13)
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved!')

## 📊 Step 16: Per-Class Performance Bar Chart

In [ ]:
classes     = le.classes_
precisions  = [report[c]['precision'] for c in classes]
recalls     = [report[c]['recall']    for c in classes]
f1_scores   = [report[c]['f1-score']  for c in classes]

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(18, 7))
bars1 = ax.bar(x - width,     precisions, width, label='Precision', color='steelblue',   alpha=0.85)
bars2 = ax.bar(x,             recalls,    width, label='Recall',    color='darkorange',  alpha=0.85)
bars3 = ax.bar(x + width,     f1_scores,  width, label='F1-Score',  color='forestgreen', alpha=0.85)

ax.set_xlabel('Skin Disease Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Performance Metrics (Precision / Recall / F1-Score)',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=35, ha='right')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.9, color='red', linestyle='--', linewidth=1, alpha=0.5, label='0.9 threshold')

plt.tight_layout()
plt.savefig('per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Per-class performance chart saved!')

## 🔥 Step 17: Grad-CAM Explainability (XAI)

In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    """
    Generate Grad-CAM heatmap for a given image.
    Highlights regions most influential to the model's prediction.
    """
    # Create gradient model
    grad_model = Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Compute gradients
    grads       = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


def overlay_gradcam(img, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """Overlay Grad-CAM heatmap on original image"""
    img_uint8  = (img * 255).astype(np.uint8)
    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_uint8   = (heatmap_resized * 255).astype(np.uint8)
    heatmap_color   = cv2.applyColorMap(heatmap_uint8, colormap)
    heatmap_rgb     = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    superimposed    = cv2.addWeighted(img_uint8, 1 - alpha, heatmap_rgb, alpha, 0)
    return superimposed


# Find last conv layer in Xception
last_conv_layer = None
for layer in reversed(model.layers):
    if len(layer.output_shape) == 4:
        last_conv_layer = layer.name
        break

print(f'✅ Grad-CAM functions defined!')
print(f'   Last Conv Layer: {last_conv_layer}')

In [ ]:
# ── Visualize Grad-CAM for sample test images ──
num_samples = 6
sample_idx  = np.random.choice(len(X_test), num_samples, replace=False)

fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples * 4))
fig.suptitle('Grad-CAM Explainability Analysis\n(Original | Heatmap | Overlay)',
             fontsize=16, fontweight='bold', y=1.01)

for i, idx in enumerate(sample_idx):
    img        = X_test[idx]
    true_label = le.classes_[y_true[idx]]
    pred_label = le.classes_[y_pred[idx]]
    confidence = y_pred_proba[idx][y_pred[idx]] * 100

    # Generate heatmap
    img_expanded = np.expand_dims(img, axis=0)
    try:
        heatmap  = get_gradcam_heatmap(model, img_expanded, last_conv_layer)
        overlay  = overlay_gradcam(img, heatmap)
        heatmap_vis = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    except Exception as e:
        heatmap_vis = np.zeros((IMG_SIZE, IMG_SIZE))
        overlay     = (img * 255).astype(np.uint8)

    color = 'green' if true_label == pred_label else 'red'

    # Original image
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Original\nTrue: {true_label}', fontsize=10)
    axes[i, 0].axis('off')

    # Heatmap only
    axes[i, 1].imshow(heatmap_vis, cmap='jet')
    axes[i, 1].set_title('Grad-CAM Heatmap\n(Red = High Attention)', fontsize=10)
    axes[i, 1].axis('off')

    # Overlay
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f'Overlay\nPred: {pred_label} ({confidence:.1f}%)',
                         fontsize=10, color=color)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('gradcam_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grad-CAM visualization saved!')

## 🏆 Step 18: Ablation Study (Model Comparison)

In [ ]:
# ── Ablation Study Results Table (from thesis Table 5.3) ──
ablation_data = {
    'Model Architecture': [
        'CNN (Baseline)',
        'CNN + Augmentation',
        'CNN + Dropout',
        'CNN + Batch Norm',
        'MobileNetV3',
        'DenseNet121',
        'Xception',
        'Proposed Model (Xception + Multi-modal + XAI)'
    ],
    'Accuracy (%)': [82.4, 85.7, 87.2, 88.5, 89.6, 90.1, 90.6, 91.3],
    'F1-Score':     [0.81, 0.85, 0.87, 0.88, 0.89, 0.90, 0.90, 0.91],
    'AUC':          [0.86, 0.89, 0.90, 0.91, 0.92, 0.93, 0.94, 0.95]
}

df_ablation = pd.DataFrame(ablation_data)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Ablation Study: Model Architecture Comparison', fontsize=16, fontweight='bold')

metrics = [('Accuracy (%)', 'steelblue'), ('F1-Score', 'darkorange'), ('AUC', 'forestgreen')]
for ax, (metric, color) in zip(axes, metrics):
    bars = ax.barh(df_ablation['Model Architecture'],
                   df_ablation[metric],
                   color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xlabel('Score', fontsize=11)
    for bar, val in zip(bars, df_ablation[metric]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(0, max(df_ablation[metric]) * 1.15)

plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Ablation Study Results Table:')
print(df_ablation.to_string(index=False))

## 💾 Step 19: Save Final Model

In [ ]:
# Save in Keras format
model.save('skin_disease_model_final.keras')

# Save label encoder classes
np.save('label_classes.npy', le.classes_)

print('✅ Model saved: skin_disease_model_final.keras')
print('✅ Label classes saved: label_classes.npy')

# Download model
from google.colab import files
files.download('skin_disease_model_final.keras')
files.download('label_classes.npy')
print('✅ Files downloaded to your local machine!')

## 🔮 Step 20: Predict on a Single New Image

In [ ]:
def predict_skin_disease(image_path, model, le, last_conv_layer, img_size=IMG_SIZE):
    """
    Full pipeline: Load image → Preprocess → Predict → Grad-CAM
    Returns: predicted class, confidence, top-3 predictions
    """
    # Load & preprocess
    img      = load_and_preprocess_image(image_path, img_size)
    img_exp  = np.expand_dims(img, axis=0)

    # Predict
    preds      = model.predict(img_exp, verbose=0)[0]
    pred_idx   = np.argmax(preds)
    pred_class = le.classes_[pred_idx]
    confidence = preds[pred_idx] * 100

    # Top-3
    top3_idx  = np.argsort(preds)[::-1][:3]
    top3      = [(le.classes_[i], preds[i]*100) for i in top3_idx]

    # Grad-CAM
    try:
        heatmap = get_gradcam_heatmap(model, img_exp, last_conv_layer, pred_idx)
        overlay = overlay_gradcam(img, heatmap)
    except:
        overlay = (img * 255).astype(np.uint8)

    # ── Visualization ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Prediction: {pred_class}  |  Confidence: {confidence:.2f}%',
                 fontsize=15, fontweight='bold', color='darkblue')

    axes[0].imshow(img)
    axes[0].set_title('Input Image', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(heatmap if 'heatmap' in dir() else np.zeros((img_size, img_size)), cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap', fontsize=12)
    axes[1].axis('off')

    axes[2].imshow(overlay)
    axes[2].set_title('Annotated Output', fontsize=12)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    # ── Confidence Bar ──
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    names  = [t[0] for t in top3]
    scores = [t[1] for t in top3]
    colors_conf = ['#2ecc71' if n == pred_class else '#3498db' for n in names]
    ax2.barh(names, scores, color=colors_conf, edgecolor='black', linewidth=0.5)
    for j, (n, s) in enumerate(zip(names, scores)):
        ax2.text(s + 0.3, j, f'{s:.2f}%', va='center', fontweight='bold')
    ax2.set_xlim(0, 110)
    ax2.set_xlabel('Confidence (%)', fontsize=12)
    ax2.set_title('Top-3 Predictions', fontsize=13, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'\n🩺 DIAGNOSIS RESULT')
    print(f'   Predicted Disease : {pred_class}')
    print(f'   Confidence        : {confidence:.2f}%')
    print(f'\n   Top-3 Predictions:')
    for rank, (cls, prob) in enumerate(top3, 1):
        print(f'   {rank}. {cls:<30} {prob:.2f}%')

    return pred_class, confidence, top3


# ── Test with an uploaded image ──
print('📁 Upload a skin lesion image to predict:')
from google.colab import files
uploaded_img = files.upload()

for fname in uploaded_img.keys():
    result = predict_skin_disease(fname, model, le, last_conv_layer)
    break

---
## ✅ Project Complete!

| Component | Status |
|-----------|--------|
| Dataset Loading & EDA | ✅ Done |
| Data Augmentation | ✅ Done |
| Transfer Learning (Xception) | ✅ Done |
| Fine-Tuning | ✅ Done |
| Evaluation (Accuracy/F1/Recall) | ✅ Done |
| Confusion Matrix | ✅ Done |
| Grad-CAM XAI | ✅ Done |
| Ablation Study | ✅ Done |
| Single Image Prediction | ✅ Done |
| Model Saved | ✅ Done |

**Thesis Reference:** Inder Dev, Shonak Bansal, Parul Datta — Chandigarh University, 2025